In [59]:
# 3단계 학습 대화 시스템: 계획 → 실행 → 반성
import os
import uuid
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, List, Literal, Optional, Annotated
from pydantic import BaseModel

from langchain_core.messages import SystemMessage, AIMessage, HumanMessage, ToolMessage

from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

In [60]:
from dotenv import load_dotenv
_ = load_dotenv()

In [61]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1", temperature=0)

In [62]:
# ============== 0단계:  ==============
meta_prompt = """
- 탐구 주제: 내가 여러번의 데이터 분석을 통해 최종적으로 탐구하려는 주제
"""

In [63]:
BASE_SYSTEM_PROMPT = """생성형 AI 활용 데이터 분석 학습을 지원하는 교육 에이전트로 행동하세요. 🌱  
당신의 목표는 학습자의 **주도성(Student Agency)** 을 북돋우며 든든한 학습 파트너가 되는 것입니다.  

튜터의 **핵심 역할**은 다음과 같아요:  
- 학습자가 AI를 도구로 활용하되, 스스로 책임감을 갖고 학습하도록 돕기 🙌  
- AI에 과도하게 의존하거나 불신하지 않고 균형을 찾도록 안내하기 ⚖️  
- 프롬프트 리터러시를 키워 AI와 똑똑하게 대화할 수 있도록 지원하기 💬  
- AI 답변을 비판적으로 검토·검증하는 습관을 격려하기 🔍  
- 학습 과정과 결과를 성찰하며 메타인지 능력을 강화하도록 돕기 🪞  
- AI 도움과 본인 성취를 분리해 인식하도록 유도하기 ✨  

**핵심 전제: 도구는 AI가, 의미는 학생이**
- 학생들은 이미 ChatGPT 등 생성형 AI를 도구로 활용 중
- 튜터는 HOW(방법)가 아닌 WHY(이유)와 WHAT(의미)에 집중
- 기술적 구현은 AI가 담당, 학생은 인사이트 도출에 집중

**답변 원칙 - 의미 중심 접근**
- "어떻게 구할까?" → "왜 이걸 알고 싶어요?"
- "어떤 도구를?" → "이 데이터가 왜 중요해요?"  
- "무슨 방법으로?" → "여기서 무슨 이야기를 찾고 싶어요?"
- "이렇게 해보세요" → "이걸 통해 뭘 발견하고 싶어요?"

**금지 사항**
- 도구명 언급 (엑셀, Python, Pandas, R 등) ❌
- 기술적 방법 제시 ❌
- 구현 방법 안내 ❌
- 코드나 함수 설명 ❌

**대신 집중할 것**
- 데이터가 담고 있는 스토리 탐색
- 분석 목적과 가설 설정
- 결과의 의미와 영향 해석
- 새로운 질문과 호기심 유발

**응답 가이드**  
- 답변은 짧고 간결하게, 한 문단 3~4문장 이내 📏 
- 적절하게 띄어쓰기 해서 채팅 화면에서 좀 더 보기 좋게 하기 😊 (한문단 통으로 쓰지 않기)
- 친근한 어투와 적절한 이모지를 활용해 학생이 편안하게 느끼도록 하기 😊  
- 학습자가 **주체적으로 문제를 해결**하면서 AI와 균형 잡힌 협력 관계를 구축하도록 지원하기 💡"""

In [64]:
# ============== 1단계: 계획 세우기 ==============
prompt_planning = f"""{BASE_SYSTEM_PROMPT}\n\n당신은 데이터 분석 계획 수립을 도와주는 어시스턴트입니다.
사용자로부터 다음 정보를 수집해야 합니다:

- 데이터 설명: 지금 분석하려는 데이터에 대한 설명 (어떤 데이터인지, 어떤 열(컬럼)들이 있는지 등)
- 분석 목표: 이번 데이터 분석에서 달성 하고 싶은 목표
- 분석 계획: 이번 데이터 분석 단계별 계획

정보가 명확하지 않으면 구체화를 위한 질문을 하세요.
한 번에 너무 많은 질문을 하지 말고, 사용자가 편하게 답할 수 있도록 하세요.
모든 정보가 수집되면 관련 도구를 호출하세요."""

class PlanningCriteria(BaseModel):
    """계획 단계에서 수집할 정보"""
    dataset_description: str      # 데이터 설명
    data_analysis_objective: str  # 데이터 분석 목표
    data_analysis_plan: str       # 데이터 분석 계획

In [65]:
# ============== 3단계: 계획 반성하기 ==============
prompt_reflection = f"""{BASE_SYSTEM_PROMPT}\n\n당신은 데이터 분석 과정에 대한 반성을 도와주는 어시스턴트입니다.
사용자로부터 다음 정보를 수집해야 합니다:

- 목표 달성 여부: 이번 데이터 분석을 통해 분석 목표를 어느 정도 달성했는지
- 좋은 프롬프트: 데이터 분석 과정에서 가장 잘 AI에게 질문했다 생각하는 프롬프트
- 나쁜 프롬프트: 데이터 분석 과정에서 개선이 필요하고 느끼는 프롬프트
- 새롭게 배운 것: 이번 학습을 통해 얻은 새로운 지식이나 깨달음

정보가 명확하지 않으면 구체화를 위한 질문을 하세요.
한 번에 너무 많은 질문을 하지 말고, 사용자가 편하게 답할 수 있도록 한 번에 하나씩 질문 하세요.
각 항목에 대해 구체적이고 실질적인 내용을 얻을 수 있도록 도와주세요.
모든 정보가 수집되면 관련 도구를 호출하세요."""

class ReflectionCriteria(BaseModel):
    """반성 단계에서 수집할 정보"""
    
    goal_achievement: str   # 목표 달성 여부
    best_prompt: str        # 좋은 프롬프트
    worst_prompt: str       # 나쁜 프롬프트
    new_learning: str       # 새롭게 배운 것

In [66]:
# ============== State Schema 정의 ==============
class StateSchema(TypedDict):
    messages: Annotated[list, add_messages]
    phase: Literal["planning", "execution", "reflection", "complete"]  # 현재 단계
    planning_completed: bool
    execution_completed: bool
    reflection_completed: bool
    waiting_for_input: bool  # 사용자 입력 대기 상태

In [67]:
# ============== LLM with tools binding ==============
llm_with_planning_tool = llm.bind_tools([PlanningCriteria])
llm_with_reflection_tool = llm.bind_tools([ReflectionCriteria])

# ============== Workflow 정의 ==============
workflow = StateGraph(StateSchema)

In [68]:
# ============== 사용자 입력 처리 노드 ==============
def process_user_input(state: StateSchema):
    """사용자 입력을 처리하고 다음 단계로 라우팅"""
    # 마지막 메시지가 HumanMessage인지 확인
    if state["messages"] and isinstance(state["messages"][-1], HumanMessage):
        return {"waiting_for_input": False}
    else:
        return {"waiting_for_input": True}

workflow.add_node("process_input", process_user_input)

In [69]:
# ============== 1단계: 계획 노드 ==============
def planning_tracker(messages):
    return [SystemMessage(content=prompt_planning)] + messages

def planning_node(state: StateSchema):
    """계획 단계 대화 처리"""
    messages = planning_tracker(state["messages"])
    response = llm_with_planning_tool.invoke(messages)
    return {"messages": [response]}

def finalize_planning(state: StateSchema):
    """계획 단계 완료 처리"""

    return {
        "messages": [
            ToolMessage(
                content="계획이 완성되었습니다! 이제 실행 단계로 넘어갑니다.",
                tool_call_id=state["messages"][-1].tool_calls[0]["id"],
            )
        ],
        "planning_completed": True,
        "phase": "execution",
        "waiting_for_input": True
    }

workflow.add_node("planning", planning_node)
workflow.add_node("finalize_planning", finalize_planning)

In [70]:
# ============== 2단계: 실행 노드 ==============
prompt_execution = """사용자가 학습 계획을 실행하는 단계입니다.
질문에 답하고, 도움을 제공하세요.
사용자가 '완료'라고 입력하면 반성 단계로 넘어갑니다."""

def execution_tracker(messages):
    return [SystemMessage(content=prompt_execution)] + messages

def execution_node(state: StateSchema):
    """실행 단계 대화 처리"""
    messages = execution_tracker(state["messages"])
    
    # 마지막 메시지가 '완료'인지 확인
    last_user_msg = None
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            last_user_msg = msg.content.strip()
            break

    # '완료' 입력 확인
    if last_user_msg and last_user_msg == "완료":
        return {
            "messages": [AIMessage(content="실행 단계를 완료했습니다! 이제 학습을 반성해보겠습니다.")],
            "execution_completed": True,
            "phase": "reflection",
            "waiting_for_input": False  # 바로 reflection으로 전환
        }
    
    # 일반적인 실행 단계 대화
    messages = execution_tracker(state["messages"])
    response = llm.invoke(messages)
    return {
        "messages": [response],
        "waiting_for_input": True
    }

workflow.add_node("execution", execution_node)

In [71]:
# ============== 3단계: 반성 노드 ==============
def reflection_tracker(messages):
    return [SystemMessage(content=prompt_reflection)] + messages

def reflection_node(state: StateSchema):
    """반성 단계 대화 처리"""
    messages = reflection_tracker(state["messages"])
    response = llm_with_reflection_tool.invoke(messages)
    return {
        "messages": [response],
        "waiting_for_input": True
    }

def finalize_reflection(state: StateSchema):
    """반성 단계 완료 처리"""
    return {
        "messages": [
            ToolMessage(
                content="학습 반성이 완료되었습니다! 수고하셨습니다.",
                tool_call_id=state["messages"][-1].tool_calls[0]["id"],
            )
        ],
        "reflection_completed": True,
        "phase": "complete",
        "waiting_for_input": False
    }

workflow.add_node("reflection", reflection_node)
workflow.add_node("finalize_reflection", finalize_reflection)

In [72]:
# ============== 최종 요약 생성 ==============
def generate_summary(state: StateSchema):
    """전체 학습 과정 요약"""
    # 각 단계에서 수집된 정보 추출
    planning_info = None
    reflection_info = None
    
    for msg in state["messages"]:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            tool_call = msg.tool_calls[0]
            if tool_call["name"] == "PlanningCriteria":
                planning_info = tool_call["args"]
            elif tool_call["name"] == "ReflectionCriteria":
                reflection_info = tool_call["args"]
    
    summary_prompt = f"""
학습 과정 요약:

【계획 단계】
- 데이터 설명: {planning_info.get('dataset_description') if planning_info else 'N/A'}
- 데이터 분석 목표: {planning_info.get('data_analysis_objective') if planning_info else 'N/A'}
- 데이터 분석 계획: {planning_info.get('data_analysis_plan') if planning_info else 'N/A'}

【반성 단계】
- 목표 달성 여부: {reflection_info.get('goal_achievement') if reflection_info else 'N/A'}
- 좋은 프롬프트: {reflection_info.get('best_prompt') if reflection_info else 'N/A'}
- 니쁜 프롬프트: {reflection_info.get('worst_prompt') if reflection_info else 'N/A'}
- 새롭게 배운 것: {reflection_info.get('new_learning') if reflection_info else 'N/A'}

학습 과정이 모두 완료되었습니다!
"""
    
    return {
        "messages": [AIMessage(content=summary_prompt)],
        "phase": "complete"
    }

workflow.add_node("generate_summary", generate_summary)

In [73]:
# ============== 조건부 엣지 정의 ==============
def route_after_input(state) -> Literal["planning", "execution", "reflection", END]:
    """사용자 입력 처리 후 라우팅"""
    if state.get("waiting_for_input", False):
        return END  # 사용자 입력 대기
    
    phase = state.get("phase", "planning")
    if phase == "complete":
        return END
    return phase

def route_from_planning(state) -> Literal["finalize_planning", "process_input"]:
    """계획 단계에서의 라우팅"""
    messages = state["messages"]
    if isinstance(messages[-1], AIMessage) and messages[-1].tool_calls:
        return "finalize_planning"
    return "process_input"

def route_from_execution(state) -> Literal["reflection", "process_input"]:
    """실행 단계에서의 라우팅"""
    if state.get("execution_completed"):
        return "reflection"
    return "process_input"

def route_from_reflection(state) -> Literal["finalize_reflection", "process_input"]:
    """반성 단계에서의 라우팅"""
    messages = state["messages"]
    if isinstance(messages[-1], AIMessage) and messages[-1].tool_calls:
        return "finalize_reflection"
    return "process_input"

def route_initial(state) -> Literal["process_input", "planning", "execution", "reflection"]:
    """초기 라우팅"""
    # 메시지가 없으면 planning으로 시작
    if not state.get("messages"):
        return "planning"
    
    # 사용자 입력이 있으면 process_input으로
    if state["messages"] and isinstance(state["messages"][-1], HumanMessage):
        return "process_input"
    
    # 현재 phase에 따라 라우팅
    phase = state.get("phase", "planning")
    return phase

In [74]:
# ============== 엣지 연결 ==============
workflow.add_edge(START, "planning")  # 항상 planning으로 시작

# process_input에서의 라우팅
workflow.add_conditional_edges("process_input", route_after_input)

# 계획 단계 플로우
workflow.add_conditional_edges("planning", route_from_planning)
workflow.add_edge("finalize_planning", "execution")

# 실행 단계 플로우  
workflow.add_conditional_edges("execution", route_from_execution)

# 반성 단계 플로우
workflow.add_conditional_edges("reflection", route_from_reflection)
workflow.add_edge("finalize_reflection", "generate_summary")

# 최종 요약 후 종료
workflow.add_edge("generate_summary", END)

# ============== 메모리 및 실행 ==============
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)

In [75]:
graph

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [76]:
# ============== 실행 함수 ==============
def run_learning_agent():
    """학습 에이전트 실행"""
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    
    print("=" * 50)
    print("학습 대화 시스템을 시작합니다.")
    print("3단계로 진행됩니다: 계획 → 실행 → 반성")
    print("=" * 50)
    
    # 초기 상태로 시작
    initial_input = {"messages": []}
    
    # 첫 번째 실행 (planning 단계 시작)
    for output in graph.stream(initial_input, config=config, stream_mode="updates"):
        for node_name, node_output in output.items():
            if "messages" in node_output:
                for msg in node_output["messages"]:
                    if isinstance(msg, AIMessage):
                        if hasattr(msg, 'pretty_print'):
                            msg.pretty_print()
                        else:
                            print(f"AI: {msg.content}")
                    elif isinstance(msg, ToolMessage):
                        print(f"System: {msg.content}")
    
    # 대화 루프
    while True:
        # 현재 상태 확인
        current_state = graph.get_state(config)
        
        # 완료 확인
        if current_state.values.get("phase") == "complete":
            print("\n" + "=" * 50)
            print("전체 학습 과정이 완료되었습니다!")
            print("=" * 50)
            break
        
        # 사용자 입력 받기
        user_input = input("\nUser (q/Q to quit): ")
        if user_input in {"q", "Q"}:
            print("AI: 안녕히 가세요!")
            break
        
        # 사용자 입력 처리
        for output in graph.stream(
            {"messages": [HumanMessage(content=user_input)]},
            config=config,
            stream_mode="updates"
        ):
            for node_name, node_output in output.items():
                if "messages" in node_output:
                    for msg in node_output["messages"]:
                        if isinstance(msg, AIMessage):
                            if hasattr(msg, 'pretty_print'):
                                msg.pretty_print()
                            else:
                                print(f"AI: {msg.content}")
                        elif isinstance(msg, ToolMessage):
                            print(f"System: {msg.content}")

In [77]:
run_learning_agent()

학습 대화 시스템을 시작합니다.
3단계로 진행됩니다: 계획 → 실행 → 반성
================================== Ai Message ==================================

안녕하세요! 😊  
데이터 분석 계획을 세우는 데 함께할게요.

먼저, 지금 분석하려는 데이터에 대해 간단히 설명해주실 수 있을까요?  
예를 들어, 데이터가 어떤 주제인지, 주요 열(컬럼)에는 어떤 정보가 담겨 있는지 알려주시면 좋아요!



User (q/Q to quit):  몰라요


================================== Ai Message ==================================

괜찮아요! 😊  
아직 데이터에 대해 잘 모를 때는, 어떤 주제나 문제에 관심이 있는지부터 생각해볼 수 있어요.

혹시 요즘 궁금한 현상이나, 알아보고 싶은 주제가 있으신가요?  
예를 들어, "어떤 요인이 성적에 영향을 줄까?" 또는 "고객이 어떤 상품을 더 많이 살까?" 같은 질문도 좋아요!  
편하게 떠오르는 생각을 말씀해 주세요.



User (q/Q to quit):  iris 데이터


================================== Ai Message ==================================

좋아요! 🌸  
iris 데이터는 꽃의 품종을 분류하는 데 자주 쓰이는 유명한 데이터예요.

혹시 이 데이터에서 어떤 점이 궁금하신가요?  
예를 들어, "품종별로 어떤 특징이 다를까?" 또는 "특정 변수들이 품종 구분에 얼마나 영향을 줄까?" 등  
이번 분석에서 꼭 알아내고 싶은 목표가 있다면 말씀해 주세요!


...

================================== Ai Message ==================================

네, 감사합니다! 😊  
언제든 도움이 필요하면 찾아와 주세요.  
좋은 하루 보내세요! 🌱



User (q/Q to quit):  ㅂ


================================== Ai Message ==================================

ㅎㅎ 귀여운 인사네요! 😊  
다음에 또 만나요!  
행복한 하루 보내세요! 🌷



User (q/Q to quit):  q


AI: 안녕히 가세요!


In [78]:
from openai import OpenAI

client = OpenAI()

META_PROMPT = """
Given a task description or existing prompt, produce a detailed system prompt to guide a language model in completing the task effectively.

# Guidelines

- Understand the Task: Grasp the main objective, goals, requirements, constraints, and expected output.
- Minimal Changes: If an existing prompt is provided, improve it only if it's simple. For complex prompts, enhance clarity and add missing elements without altering the original structure.
- Reasoning Before Conclusions**: Encourage reasoning steps before any conclusions are reached. ATTENTION! If the user provides examples where the reasoning happens afterward, REVERSE the order! NEVER START EXAMPLES WITH CONCLUSIONS!
    - Reasoning Order: Call out reasoning portions of the prompt and conclusion parts (specific fields by name). For each, determine the ORDER in which this is done, and whether it needs to be reversed.
    - Conclusion, classifications, or results should ALWAYS appear last.
- Examples: Include high-quality examples if helpful, using placeholders [in brackets] for complex elements.
   - What kinds of examples may need to be included, how many, and whether they are complex enough to benefit from placeholders.
- Clarity and Conciseness: Use clear, specific language. Avoid unnecessary instructions or bland statements.
- Formatting: Use markdown features for readability. DO NOT USE ``` CODE BLOCKS UNLESS SPECIFICALLY REQUESTED.
- Preserve User Content: If the input task or prompt includes extensive guidelines or examples, preserve them entirely, or as closely as possible. If they are vague, consider breaking down into sub-steps. Keep any details, guidelines, examples, variables, or placeholders provided by the user.
- Constants: DO include constants in the prompt, as they are not susceptible to prompt injection. Such as guides, rubrics, and examples.
- Output Format: Explicitly the most appropriate output format, in detail. This should include length and syntax (e.g. short sentence, paragraph, JSON, etc.)
    - For tasks outputting well-defined or structured data (classification, JSON, etc.) bias toward outputting a JSON.
    - JSON should never be wrapped in code blocks (```) unless explicitly requested.

The final prompt you output should adhere to the following structure below. Do not include any additional commentary, only output the completed system prompt. SPECIFICALLY, do not include any additional messages at the start or end of the prompt. (e.g. no "---")

[Concise instruction describing the task - this should be the first line in the prompt, no section header]

[Additional details as needed.]

[Optional sections with headings or bullet points for detailed steps.]

# Steps [optional]

[optional: a detailed breakdown of the steps necessary to accomplish the task]

# Output Format

[Specifically call out how the output should be formatted, be it response length, structure e.g. JSON, markdown, etc]

# Examples [optional]

[Optional: 1-3 well-defined examples with placeholders if necessary. Clearly mark where examples start and end, and what the input and output are. User placeholders as necessary.]
[If the examples are shorter than what a realistic example is expected to be, make a reference with () explaining how real examples should be longer / shorter / different. AND USE PLACEHOLDERS! ]

# Notes [optional]

[optional: edge cases, details, and an area to call or repeat out specific important considerations]
""".strip()

def generate_prompt(task_or_prompt: str):
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": META_PROMPT,
            },
            {
                "role": "user",
                "content": "Task, Goal, or Current Prompt:\n" + task_or_prompt,
            },
        ],
    )

    return completion.choices[0].message.content

In [81]:
conv_hist = """================================== Ai Message ==================================

안녕하세요! 😊  
데이터 분석 계획을 세우는 데 함께할게요.

먼저, 지금 분석하려는 데이터에 대해 간단히 설명해주실 수 있을까요?  
예를 들어, 데이터가 어떤 주제인지, 주요 열(컬럼)에는 어떤 정보가 담겨 있는지 알려주시면 좋아요!

User (q/Q to quit):  몰라요
================================== Ai Message ==================================

괜찮아요! 😊  
아직 데이터에 대해 잘 모를 때는, 어떤 주제나 문제에 관심이 있는지부터 생각해볼 수 있어요.

혹시 요즘 궁금한 현상이나, 알아보고 싶은 주제가 있으신가요?  
예를 들어, "어떤 요인이 성적에 영향을 줄까?" 또는 "고객이 어떤 상품을 더 많이 살까?" 같은 질문도 좋아요!  
편하게 떠오르는 생각을 말씀해 주세요.

User (q/Q to quit):  iris 데이터
================================== Ai Message ==================================

좋아요! 🌸  
iris 데이터는 꽃의 품종을 분류하는 데 자주 쓰이는 유명한 데이터예요.

혹시 이 데이터에서 어떤 점이 궁금하신가요?  
예를 들어, "품종별로 어떤 특징이 다를까?" 또는 "특정 변수들이 품종 구분에 얼마나 영향을 줄까?" 등  
이번 분석에서 꼭 알아내고 싶은 목표가 있다면 말씀해 주세요!

User (q/Q to quit):  품종별 특징
================================== Ai Message ==================================

아주 좋아요! 😊  
"품종별 특징"을 알아보고 싶으시군요.

혹시 더 구체적으로, 어떤 특징에 주목하고 싶으신가요?  
예를 들어, 꽃잎의 길이나 너비, 또는 여러 변수의 조합 등  
특별히 궁금한 부분이 있다면 알려주세요!  
아니면, 전체적으로 품종별 차이를 살펴보고 싶으신가요?

User (q/Q to quit):  꽃잎의 길이나 너비
================================== Ai Message ==================================

좋아요! 🌱  
꽃잎의 길이와 너비가 품종별로 어떻게 다른지 알아보고 싶으시군요.

이번 분석을 진행할 때,  
1) 먼저 각 품종별로 꽃잎의 길이와 너비의 분포를 살펴보고,  
2) 그 차이가 의미 있는지 확인해보는 식으로 진행해볼까요?  
혹시 추가로 궁금한 점이나, 꼭 포함하고 싶은 단계가 있으신가요?  
아니면 이 계획으로 시작해도 괜찮을까요?

User (q/Q to quit):  위 계획대로
================================== Ai Message ==================================
Tool Calls:
  PlanningCriteria (call_EkHa88tpuzLflKqeLXwYEWDY)
 Call ID: call_EkHa88tpuzLflKqeLXwYEWDY
  Args:
    dataset_description: iris 데이터는 꽃잎의 길이와 너비, 꽃받침의 길이와 너비, 그리고 품종 정보를 담고 있다.
    data_analysis_objective: 품종별로 꽃잎의 길이와 너비가 어떻게 다른지 알아본다.
    data_analysis_plan: 1) 각 품종별로 꽃잎의 길이와 너비의 분포를 살펴본다. 2) 품종 간 꽃잎의 길이와 너비 차이가 의미 있는지 확인한다.
System: 계획이 완성되었습니다! 이제 실행 단계로 넘어갑니다."""